# ⚖️ Day 01a: Load Balancers & CDN

---

## 🎯 What You'll Master Today
- Load balancer types (L4 vs L7)
- LB algorithms (round robin, weighted, least connections, IP hash)
- CDN — push vs pull, edge caching
- When to use what

**The analogy:** A load balancer is the HOSTESS at a busy restaurant — she decides which table (server) gets the next customer.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  LOAD BALANCER — THE BIG PICTURE                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Without LB:          With LB:                                   ║
║                                                                   ║
║  Client → Server      Client → [LB] → Server 1                 ║
║  (single point         ↑         │  → Server 2                  ║
║   of failure!)                   │  → Server 3                  ║
║                                  └──→ Server N                  ║
║                                                                   ║
║  Benefits:                                                       ║
║  ✅ No single point of failure                                   ║
║  ✅ Horizontal scaling (add more servers)                        ║
║  ✅ SSL termination (offload crypto)                             ║
║  ✅ Health checks (remove dead servers)                          ║
║  ✅ Session persistence (sticky sessions)                        ║
║                                                                   ║
║  Where to place LBs:                                            ║
║  Internet → [LB] → Web Servers → [LB] → App Servers            ║
║            → [LB] → DB Servers                                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. L4 vs L7 Load Balancers
# ============================================================

print("""
L4 vs L7 LOAD BALANCERS
═══════════════════════

┌──────────────┬─────────────────────┬─────────────────────┐
│ Feature      │ L4 (Transport)      │ L7 (Application)    │
├──────────────┼─────────────────────┼─────────────────────┤
│ Layer        │ TCP/UDP             │ HTTP/HTTPS           │
│ Sees         │ IP + Port           │ Headers, URL, Body   │
│ Routes by    │ Connection (IP hash)│ Content (URL path)   │
│ Speed        │ ⚡ Very fast         │ Slower (inspects)    │
│ Features     │ Basic               │ SSL, compression     │
│ Use case     │ TCP traffic, gaming │ Web apps, APIs       │
│ Examples     │ AWS NLB, HAProxy L4 │ AWS ALB, Nginx, HAP  │
└──────────────┴─────────────────────┴─────────────────────┘

L7 can do content-based routing:
  /api/*     → API servers
  /images/*  → Image servers  
  /video/*   → Video servers

💡 Interview insight: "I'd use L7 for microservices (route by path)
   and L4 for raw TCP like database or game servers."
""")

In [ ]:
# ============================================================
# 2. Load Balancing Algorithms
# ============================================================

import random
from collections import Counter

servers = ["S1", "S2", "S3"]

# --- Round Robin ---
def round_robin(n_requests):
    return [servers[i % len(servers)] for i in range(n_requests)]

print("Round Robin:", round_robin(9))

# --- Weighted Round Robin ---
def weighted_round_robin(n_requests, weights):
    """weights = {server: weight}"""
    pool = []
    for s, w in weights.items():
        pool.extend([s] * w)
    return [pool[i % len(pool)] for i in range(n_requests)]

weights = {"S1": 5, "S2": 3, "S3": 2}  # S1 is 2.5× more powerful
result = weighted_round_robin(20, weights)
print(f"\nWeighted RR ({weights}):")
print(f"  Distribution: {dict(Counter(result))}")

# --- Least Connections ---
def least_connections(requests, initial_load=None):
    load = initial_load or {s: 0 for s in servers}
    assignments = []
    for _ in requests:
        chosen = min(load, key=load.get)
        load[chosen] += 1
        assignments.append(chosen)
        # Simulate random completion
        if random.random() > 0.7:
            done = random.choice(list(load.keys()))
            load[done] = max(0, load[done] - 1)
    return assignments

print(f"\nLeast Connections: distributes to least-busy server")
print(f"  Best for: varied request durations (long-running tasks)")

# --- IP Hash ---
def ip_hash(client_ips):
    return [servers[hash(ip) % len(servers)] for ip in client_ips]

ips = ["192.168.1.1", "10.0.0.5", "192.168.1.1", "172.16.0.1", "10.0.0.5"]
print(f"\nIP Hash: same IP → same server (session affinity)")
for ip, server in zip(ips, ip_hash(ips)):
    print(f"  {ip} → {server}")

In [ ]:
# ============================================================
# 3. Algorithm Decision Guide
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  WHICH LB ALGORITHM?                                          ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Scenario                       │ Algorithm                   ║
║  ───────────────────────────────│──────────────────────────   ║
║  All servers identical          │ Round Robin                 ║
║  Servers have diff capacity     │ Weighted Round Robin        ║
║  Requests take variable time    │ Least Connections           ║
║  Need session affinity          │ IP Hash                     ║
║  Geographically distributed     │ Geo-based / Latency-based  ║
║  Random but statistically fair  │ Random                      ║
║                                                                ║
║  💡 Most production systems use Least Connections or           ║
║     Weighted Round Robin with health checks                   ║
║                                                                ║
║  LB redundancy: Active-Passive or Active-Active               ║
║    ┌────┐  heartbeat  ┌────┐                                  ║
║    │ LB1│◄───────────►│ LB2│ (standby takes over if active   ║
║    └────┘              └────┘  fails — floating IP / VIP)     ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# 4. CDN — Content Delivery Network
# ============================================================

print("""
CDN — CONTENT DELIVERY NETWORK
═══════════════════════════════

Without CDN:                    With CDN:
                                
User (India) ──5000km──► US     User (India) ──50km──► Edge (Mumbai)
Latency: 200ms                  Latency: 10ms

CDN = geographically distributed cache servers
      that serve static content from the nearest edge.


PUSH vs PULL CDN:
┌──────────────┬───────────────────────┬───────────────────────┐
│ Feature      │ Push CDN              │ Pull CDN              │
├──────────────┼───────────────────────┼───────────────────────┤
│ How it works │ You upload to CDN     │ CDN fetches on 1st    │
│              │ when content changes  │ request, then caches  │
│ Control      │ You decide what/when  │ CDN decides via TTL   │
│ Storage cost │ Higher (all pushed)   │ Lower (on-demand)     │
│ Best for     │ Small, rarely         │ Heavy traffic sites   │
│              │ changing content      │ with lots of content  │
│ Examples     │ Firmware updates      │ Netflix, news sites   │
└──────────────┴───────────────────────┴───────────────────────┘

CDN Cache Flow (Pull):
  1. User requests image.jpg
  2. Edge server checks cache → MISS
  3. Edge fetches from origin server
  4. Edge caches + serves to user
  5. Next user request → cache HIT → served from edge
  6. After TTL expires → edge revalidates with origin

What CDN caches:
  ✅ Static: images, CSS, JS, fonts, videos
  ✅ HTML pages (with careful cache-control headers)
  ❌ Dynamic API responses (usually not cached)
  ⚠️  Personalized content (use edge compute)
""")

# CDN providers reference
providers = {
    "CloudFront": "AWS — integrates with S3, EC2",
    "Cloudflare": "Also DDoS protection, DNS, WAF",
    "Akamai":     "Largest network, enterprise focus",
    "Fastly":     "Edge compute (Wasm), real-time purge",
}
print("\nCDN Providers:")
for name, desc in providers.items():
    print(f"  {name:12s} — {desc}")

In [ ]:
# ============================================================
# 5. Consistent Hashing (for LB + CDN distribution)
# ============================================================

import hashlib

class ConsistentHash:
    """
    Consistent hashing — used by CDNs, caches, and load balancers.
    When a server is added/removed, only K/N keys need to move.
    """
    def __init__(self, nodes=None, replicas=3):
        self.replicas = replicas  # virtual nodes per physical node
        self.ring = {}           # hash → node mapping
        self.sorted_keys = []     # sorted hash values
        if nodes:
            for node in nodes:
                self.add_node(node)
    
    def _hash(self, key):
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % 360
    
    def add_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}:{i}")
            self.ring[h] = node
            self.sorted_keys.append(h)
        self.sorted_keys.sort()
    
    def remove_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}:{i}")
            del self.ring[h]
            self.sorted_keys.remove(h)
    
    def get_node(self, key):
        if not self.ring:
            return None
        h = self._hash(key)
        for k in self.sorted_keys:
            if k >= h:
                return self.ring[k]
        return self.ring[self.sorted_keys[0]]  # wrap around

# Demo
ch = ConsistentHash(["Server-A", "Server-B", "Server-C"])

test_keys = ["user:1001", "user:1002", "img:abc.jpg", "video:xyz"]
print("Initial distribution:")
for key in test_keys:
    print(f"  {key:20s} → {ch.get_node(key)}")

print("\nRemoving Server-B...")
ch.remove_node("Server-B")
print("After removal:")
for key in test_keys:
    print(f"  {key:20s} → {ch.get_node(key)}")

print("\n💡 Only some keys moved! This is the magic of consistent hashing.")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | L4 vs L7 load balancer? | L4 = TCP level (fast, simple). L7 = HTTP level (content routing, SSL termination) |
| 2 | How to handle LB failure? | Active-passive with floating IP. Health checks via heartbeat |
| 3 | Push vs Pull CDN? | Push = you upload. Pull = CDN fetches on miss. Pull better for high-traffic |
| 4 | What is consistent hashing? | Hash ring. Add/remove node → only K/N keys move. Used in Dynamo, Cassandra, CDNs |
| 5 | Sticky sessions vs stateless? | Sticky = IP hash (simple but uneven). Stateless = externalize state to Redis (preferred) |
| 6 | Choose LB algo for scenario | Variable duration → least connections. Same capacity → round robin. Sessions → IP hash |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • L4 = TCP-level (fast), L7 = HTTP-level (smart)       │
## │  • Weighted RR / Least Connections most common           │
## │  • CDN = edge caching for static content                 │
## │  • Pull CDN for most cases, Push for small static sets   │
## │  • Consistent hashing minimizes key redistribution       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Caching — Redis, Cache-Aside & Invalidation**